In [93]:
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.optim import Adam
from torchvision import datasets, transforms


In [94]:
print('==> Preparing data.............................')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.datasets import MNIST, CIFAR10, CIFAR100, SVHN

import os
import torch
from torch import nn,optim
import torch.nn.functional as F
from torchvision import datasets, transforms

from time import perf_counter

import  numpy as np
import torch.utils.data as Data

from torch.utils.data import Dataset, DataLoader

class Safeman(Dataset):
    
    def __init__(self, data,targets):
        super(Safeman, self).__init__()
        self.data = data
        self.targets = targets
        
     
    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        img, target = self.data[idx], self.targets[idx]
        return img, target

class Safeman_Filter(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)
        
class Safeman_FilterB(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        new_targets = []
        for i in range(len(targets)):
            if targets[i] in known:
                new_targets.append(0)
            else:
                new_targets.append(1)
        self.targets = np.array(new_targets)
        self.data = self.data

class Safeman_FilterC(Safeman):
    
    def __Filter__(self, trainknown):
        train_class_num=len(trainknown)
        for i in range(0,len(self.targets)) :
            if self.targets[i]>train_class_num:
                self.targets[i] = train_class_num
        self.data = self.data

        
        
class Safeman_FilterF(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                dd = known.index(targets[i])
                if dd == 4:
                    new_targets.append(0)
                else:
                    new_targets.append(1)                   
                    
                #new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)   


        
def setup_seed(seed):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(8)



known=[0, 1, 2,3,4,5,6,7]
unknown=[ 5,6,7]





X_train0 = np.load('./TONdataset/x_train_iot1028+1del.npy')
y_train1 = np.load('./TONdataset/y_train_iot1028+1del.npy')
X_final_test0 = np.load('./TONdataset/x_test_iot1028+1del.npy' )
y__final_test1 = np.load('./TONdataset/y_test_iot1028+1del.npy')





X_train1=[]
X_final_test1=[]

for i in range(len(y_train1)):
    a = np.resize(X_train0[i], (21))
    X_train1 += [a]
    
for j in range(len(y__final_test1)):
    b = np.resize(X_final_test0[j], (21))
    X_final_test1 += [b]

i=0
j=0



x_train, x_test, y_train,y_test = torch.Tensor(X_train1), torch.Tensor(X_final_test1), torch.from_numpy(y_train1), torch.from_numpy(y__final_test1)

print(x_train.shape, x_test.shape, y_train.shape,y_test.shape)

train_dataset = Data.TensorDataset(x_train, y_train)
train_dataset.data = train_dataset.tensors[0]
train_dataset.targets = train_dataset.tensors[1]



test_dataset = Data.TensorDataset(x_test, y_test)
test_dataset.data = test_dataset.tensors[0]
test_dataset.targets = test_dataset.tensors[1]



labels =['backdoor', 'ddos', 'dos', 'injection', 'normal', 'password', 'scanning', 'xss']

train_dataset.classes = labels
test_dataset.classes = labels

train_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}
test_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}

num_class=len(labels)

b_s=256




trainset = Safeman_Filter(data=train_dataset.data,targets=train_dataset.targets)
print('All down Train Data:', len(trainset))
trainset.__Filter__(known=known)

#0930
train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real train Data:', len(trainset))



testsetA = Safeman_FilterF(data=test_dataset.data,targets=test_dataset.targets)
print('All testsetA Data:', len(testsetA))
testsetA.__Filter__(known=known)


test_loader_A = torch.utils.data.DataLoader(
    testsetA, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real testsetA Data:', len(testsetA))


print("done!")

==> Preparing data.............................
torch.Size([242000, 21]) torch.Size([48000, 21]) torch.Size([242000]) torch.Size([48000])
All down Train Data: 242000
Real train Data: 242000
All testsetA Data: 48000
Real testsetA Data: 48000
done!


In [95]:
unique,counts = np.unique(trainset.targets,return_counts=True)
print(unique, counts)

[0 1 2 3 4 5 6 7] [ 16000  16000  16000  16000 130000  16000  16000  16000]


In [96]:
X_trainset_data=trainset.data
X_trainset_targets=trainset.targets

In [97]:
unique,counts = np.unique(X_trainset_targets,return_counts=True)
print(unique, counts)

[0 1 2 3 4 5 6 7] [ 16000  16000  16000  16000 130000  16000  16000  16000]


In [98]:
X_trainset_targets

array([6, 6, 6, ..., 0, 0, 0])

In [99]:
count=[10, 10, 10, 10, 130000, 10, 10, 10]
num_class=8
lists = [[] for i in range(num_class)]
y_train_temp=[]
x_train_temp=[]

In [100]:
for i in range(len(X_trainset_targets)):
    if len(lists[X_trainset_targets[i]])<count[X_trainset_targets[i]]:
        lists[X_trainset_targets[i]].append(X_trainset_targets[i])   
        y_train_temp+=[X_trainset_targets[i]]
        a = np.resize(X_trainset_data[i], (21))
        x_train_temp += [a]


In [101]:
unique,counts = np.unique(y_train_temp,return_counts=True)
print(unique, counts)

[0 1 2 3 4 5 6 7] [    10     10     10     10 130000     10     10     10]


In [102]:
x_train_temp[1]

array([0.0000000e+00, 4.0047044e-01, 6.5473884e-01, 3.0163100e-01,
       2.5041966e-02, 5.0000000e-01, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 5.0000000e-01, 0.0000000e+00, 1.3197836e-05,
       9.2891190e-07, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00], dtype=float32)

In [103]:
x_train2, y_train2 = torch.Tensor(x_train_temp), torch.Tensor(y_train_temp)

print(x_train2.shape, y_train2.shape)

train_dataset2 = Data.TensorDataset(x_train2, y_train2)
train_dataset2.data = train_dataset2.tensors[0]
train_dataset2.targets = train_dataset2.tensors[1]




trainset2 = Safeman_Filter(data=train_dataset2.data,targets=train_dataset2.targets)
print('All down Train Data:', len(trainset2))
trainset2.__Filter__(known=known)



train_loader2 = torch.utils.data.DataLoader(
    trainset2, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real train Data2:', len(trainset2))

torch.Size([130070, 21]) torch.Size([130070])
All down Train Data: 130070
Real train Data2: 130070


In [104]:
from collections import Counter
from imblearn.over_sampling import SMOTE,BorderlineSMOTE
import time
time_start = time.time()

smo = SMOTE(random_state=42) 

In [105]:
Counter(trainset2.targets)

Counter({6: 10, 2: 10, 3: 10, 1: 10, 4: 130000, 5: 10, 7: 10, 0: 10})

In [106]:

guo=6000*2

x_train_temp1, y_train_temp1 =  BorderlineSMOTE(sampling_strategy={6: guo, 2: guo, 3: guo, 1: guo, 5: guo, 7: guo, 0: guo},random_state=42).fit_resample(trainset2.data.numpy(), trainset2.targets) 


In [107]:
Counter(y_train_temp1)

Counter({6: 10,
         2: 12000,
         3: 12000,
         1: 12000,
         4: 130000,
         5: 12000,
         7: 10,
         0: 10})

In [108]:
x_train_temp1.shape

(178030, 21)

In [109]:
smotemal=[]
ysmotemal=[]
for i in range(len(y_train_temp1)):
    if y_train_temp1[i]!=4:
        smotemal+=[x_train_temp1[i]]
        ysmotemal+=[y_train_temp1[i]]

In [110]:
smotemal2=torch.squeeze(torch.tensor(smotemal)).numpy()

In [111]:
smotemal2.shape

(48030, 21)

In [112]:
np.save("./data/BorderlineSMOTEmal46-iot-multi-sa2-f.npy",smotemal2)
print("smotemal2",smotemal2.shape)

smotemal2 (48030, 21)


In [113]:
smotemal3=torch.squeeze(torch.tensor(ysmotemal)).numpy()

In [114]:
smotemal3.shape

(48030,)

In [115]:
np.save("./data/BorderlineSMOTEmal46-iot-multi-label-sa2-f.npy",smotemal3)
print("smotemal3",smotemal3.shape)

smotemal3 (48030,)
